# Location Selection with E-NAUTILUS: Part 2
_Running of E-NAUTILUS for decision making, and presentation of results_


In [26]:
import numpy as np
import pandas as pd
import polars as pl
import pickle
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Load results from previous session

In [27]:
file_name = "data/pf_test.pkl"

output = open(file_name, 'rb')
prev_session = pickle.load(output)

raw_ref_pf = prev_session["pf"]
prob = prev_session["prob"]
sites = prev_session["sites"]
cities = prev_session["cities"]
cities_adj2sites = prev_session["cities_adj2sites"]


## Load reference front and problem 

In [28]:

output_flat = np.array(raw_ref_pf).flatten()

def process_lists(dict2conv):
    return {key: np.array(dict2conv[key]).flatten().tolist() for key in dict2conv.keys()}

# TODO include constraints in here too
output_dict = [
    output.optimal_objectives | 
    process_lists(output.optimal_variables) 
    for output in output_flat]

nd_df = pl.DataFrame(output_dict)

nd_df = nd_df.unique(subset=("f_1", "f_2", "f_3", "f_4"))

nd_df = nd_df.with_columns([
    (-pl.col("f_1")).alias("f_1_min"),
    (pl.col("f_2")).alias("f_2_min"),
    (pl.col("f_3")).alias("f_3_min"),
    (-pl.col("f_4")).alias("f_4_min")
])

nadir_point = {
  "f_1": float(nd_df["f_1"].min()),
  "f_2": float(nd_df["f_2"].max()),
  "f_3": float(nd_df["f_3"].max()),
  "f_4": float(nd_df["f_4"].min())
}

nd_df = nd_df.unique()

display(nd_df)

print(f"Nadir point: {nadir_point}")
print(f"Nadir point (problem): {prob.get_nadir_point()}")
print(f"Idedal point (problem): {prob.get_ideal_point()}")


reachable_indices = list(range(len(nd_df)))  # everything reachable from nadir


f_1,f_2,f_3,f_4,sv,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
181.999984,16.025753,3730.602302,129805.730961,"[1.0, 1.0, … 1.0]","[0.932682, 0.614716, … 3.1153e-17]",[8.2742e-8],-181.999984,16.025753,3730.602302,-129805.730961
149.193121,3.064379,2573.308323,132448.428596,"[2.8193e-7, 0.213891, … 1.0]","[0.948653, 0.745177, … 0.0]",[0.180258],-149.193121,3.064379,2573.308323,-132448.428596
149.193121,3.064379,2573.310022,132447.348907,"[2.8193e-7, 0.213917, … 1.0]","[0.948662, 0.745051, … 0.0]",[0.180258],-149.193121,3.064379,2573.310022,-132447.348907
105.434585,0.000007,1614.739363,83105.725428,"[2.3555e-7, 4.6973e-7, … 0.999998]","[0.932431, 0.62387, … 0.0]",[0.420689],-105.434585,0.000007,1614.739363,-83105.725428
181.999984,16.03002,3731.074046,130461.337832,"[1.0, 1.0, … 1.0]","[0.943322, 0.615155, … 0.0]",[157.816155],-181.999984,16.03002,3731.074046,-130461.337832
…,…,…,…,…,…,…,…,…,…,…
117.267636,1.760779,1991.944837,141083.987198,"[0.037348, 0.044381, … 0.775108]","[1.0, 1.0, … 8.2250e-11]",[0.545395],-117.267636,1.760779,1991.944837,-141083.987198
105.414191,0.000008,1615.169482,93333.294485,"[2.4010e-7, 4.8167e-7, … 0.999998]","[0.999994, 0.999962, … 0.0]",[-0.579199],-105.414191,0.000008,1615.169482,-93333.294485
127.55971,1.804368,2186.011551,141083.987228,"[0.049985, 0.034165, … 0.847832]","[1.0, 1.0, … 0.0]",[0.302718],-127.55971,1.804368,2186.011551,-141083.987228


Nadir point: {'f_1': 1.4658469922432189e-05, 'f_2': 16.030020341101803, 'f_3': 3731.074097904416, 'f_4': 0.023575287154127347}
Nadir point (problem): {'f_1': 0, 'f_2': 17, 'f_3': 3838.3199999999997, 'f_4': 0}
Idedal point (problem): {'f_1': 182, 'f_2': 0, 'f_3': 0, 'f_4': 161142}


## Helper functions

In [29]:
# Function to determine marker size based on population
def get_marker_size(population):
    return max(5, population / 1000)  # Adjust the divisor to scale marker size

def create_color_dict(cities, ev_cities, cc): 
    marker_color = {}
    for city in cities.loc[:,"city"]: 
        if city in ev_cities: 
            marker_color[city] = "orange"
        elif city in cc: 
            marker_color[city] = "yellow"
        else: 
            marker_color[city] = "grey"

    return marker_color

def select_point(results, sol_id): 

    return {
        "f_1": int(results.loc[sol_id, "Total patients served"]),
        "f_2": int(results.loc[sol_id, "# of visited sites that are under-attended"]),
        "f_3": float(results.loc[sol_id, "Total costs ($)"]),
        "f_4": float(results.loc[sol_id, "Population with access (%)"]/100.0)
        }


def clean_results(raw_results, intermediate_point=True): 
    # Transform objectives
    if intermediate_point: 
        results = pd.DataFrame(raw_results.intermediate_points)
    else:
        results = pd.DataFrame(raw_results.optimal_objectives)

    results = results.rename(columns={
                    "f_1": "Total patients served", 
                    "f_2": "# of visited sites that are under-attended", 
                    "f_3": "Total costs ($)", 
                    "f_4": "Population with access (%)"})
    results[["Total patients served"]] =  results[["Total patients served"]].astype(int)
    results[["Population with access (%)","# of visited sites that are under-attended"]] = (results[["Population with access (%)", "# of visited sites that are under-attended"]]*100.0).round(2)
    results[["Total costs ($)"]] = (results[["Total costs ($)"]]).round(2)

    results.index.name = "Solution ID"

    return results

## Calculate HV

In [43]:
from pymoo.indicators.hv import HV

ref_point = np.array(list(nadir_point.values()))
ref_point[0] = -ref_point[0]
ref_point[3] = -ref_point[3]

pf = nd_df.select(["f_1", "f_2", "f_3", "f_4"]).to_numpy()

pf[:,0] = -pf[:,0]
pf[:,3] = -pf[:,3]

ind = HV(ref_point=ref_point)

int(ind(pf))

715871411023

## Run eNAUTILUS 
### Round 1
We're going to generate some solutions. They will be poor at first, but you and the computer will slowly find the best solution that fulfills your goals and preferences. 


In [5]:
# Initialize a first solution 
from desdeo.mcdm.enautilus import enautilus_step
from desdeo.mcdm.enautilus import enautilus_get_representative_solutions

current_iter = 0
selected_point = nadir_point
total_iterations = 3
display(f"Starting with point {selected_point}")

prob.get_ideal_point()


raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you prefer?")
display(results)

"Starting with point {'f_1': 1.4658469922432189e-05, 'f_2': 16.030020341101803, 'f_3': 3731.074097904416, 'f_4': 0.023575287154127347}"

number of iterations left: 3


'Which solution to do you prefer?'

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,0,1068.67,2489.48,14989.12
1,38,1068.67,3103.15,3090445.02
2,60,1602.35,3730.35,4566009.62


### Round 2 

In [6]:
chosen_solution = 1

In [8]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")
display(raw_results)
results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)
display("Results:")



{'f_1': 76, 'f_2': 53400, 'f_3': 2475.22, 'f_4': 61808.8768}
number of iterations left: 1


ENautilusResult(current_iteration=3, iterations_left=0, intermediate_points=[{'f_1': 0.5077864619166989, 'f_2': 6.109644976182618e-06, 'f_3': 6.291207855044419, 'f_4': 449.62635789347786}, {'f_1': 114.64174924828134, 'f_2': 6.329179268211206e-07, 'f_3': 1847.2873978274263, 'f_4': 92713.30336944251}, {'f_1': 181.9999952018061, 'f_2': 16.01034880589753, 'f_3': 3728.899191524913, 'f_4': 136980.24140090437}], reachable_best_bounds=[{'f_1': 0.5077864619166989, 'f_2': 6.109644976182618e-06, 'f_3': 6.291207855044419, 'f_4': 449.62635789347786}, {'f_1': 114.64174924828134, 'f_2': 6.329179268211206e-07, 'f_3': 1847.2873978274263, 'f_4': 92713.30336944251}, {'f_1': 181.9999952018061, 'f_2': 16.01034880589753, 'f_3': 3728.899191524913, 'f_4': 136980.24140090437}], reachable_worst_bounds=[{'f_1': 0.5077864619166989, 'f_2': 6.109644976182618e-06, 'f_3': 6.291207855044419, 'f_4': 449.62635789347786}, {'f_1': 114.64174924828134, 'f_2': 6.329179268211206e-07, 'f_3': 1847.2873978274263, 'f_4': 92713.30

'Which solution to do you find most preferable?'

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,0,0.00,6.29,44962.64
1,114,0.00,1847.29,9271330.34
2,181,1601.03,3728.90,13698024.14


'Results:'

### Round 3

In [9]:
chosen_solution = 1

In [10]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)




{'f_1': 114, 'f_2': 0, 'f_3': 1847.29, 'f_4': 92713.3034}


ZeroDivisionError: division by zero

## Display final result

In [10]:
final_chosen_solution = 2

In [11]:
# Get final solution 
solutions = enautilus_get_representative_solutions(prob, raw_results, nd_df) 
solution = solutions[final_chosen_solution]
results = clean_results(solution, intermediate_point=False)

display(results)
 


,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,181,1601.03,3728.9,13698024.14


### Result postprocessing

In [12]:
# Post process result...
raw_sites = solution.optimal_variables['sv'][0].to_list()
raw_sites = [[bool(e) for e in raw_sites]]

raw_coverage = solution.optimal_variables['cover'][0].to_list()
raw_coverage = [[bool(c) for c in raw_coverage]]

sites_visited = []
for evb in raw_sites: 
    sites_visited.append("\n".join(sites.loc[evb, "site_id"].values))

cities_covered = [] 
for cc in raw_coverage: 
    cities_covered.append("\n".join(cities.loc[cc,"city"].values))

results["Sites Visited"] = sites_visited
results["Cities covered"] = cities_covered

results

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%),Sites Visited,Cities covered
Solution ID,,,,,,
0,181,1601.03,3728.9,13698024.14,lima-mercy-thrift-store\nlima-lima-library\nke...,Ada\nAlger\nBluffton\nCairo\nCaledonia\nCarey\...


### Map preprocessing

In [13]:
# Process for the map
sites_in_cities = sites.loc[raw_sites[0],:].groupby("city").agg({"site_pretty": lambda e : '<br>'.join(e)})
sites_in_cities = sites_in_cities.to_dict()['site_pretty']
sites_in_cities

# cc cities covered
cc = set(cities_covered[0].split('\n'))

site_cities = list(sites.loc[raw_sites[0], "city"])
marker_colors = create_color_dict(cities, site_cities, cc)


## Site coverage dictionary 
site2city_mat = cities_adj2sites[raw_sites[0]].astype(bool)

site2city_dict = {}
for (c,city) in enumerate(site_cities): 
    site2city_dict[city] = set(cities.loc[site2city_mat[c,],"city"]) - {city}

adjacent_sites = {}
# Record what sites are near other cities
for site_city in site2city_dict.keys(): 
    adj_cities = site2city_dict[site_city]
    from_name = cities.loc[cities.loc[:,"city"] == site_city,["city"]].values.tolist()[0][0]

    for adj_city in adj_cities: 
        to_name = cities.loc[cities.loc[:,"city"] == adj_city ,["city"]].values.tolist()[0][0]

        if to_name not in adjacent_sites.keys(): 
            adjacent_sites[to_name] = {from_name}
        else: 
            adjacent_sites[to_name] = adjacent_sites[to_name].union({from_name})



## Render map

In [14]:
# Render map

# Create a base map
m = folium.Map(location=[cities['lat'].mean(), 
                         cities['long'].mean()], 
                         zoom_start=7) 

# Draw lines between 
for site_city in site2city_dict.keys(): 
    adj_cities = site2city_dict[site_city]
    from_loc = cities.loc[cities.loc[:,"city"] == site_city,["lat", "long"]].values.tolist()

    for adj_city in adj_cities: 
        to_loc = cities.loc[cities.loc[:,"city"] == adj_city ,["lat", "long"]].values.tolist()
        folium.PolyLine(
            locations=[to_loc[0], from_loc[0]],
            color="black"
        ).add_to(m)

# Set bounds
sw = cities.loc[:,['lat', 'long']].min().values.tolist()
ne = cities.loc[:,['lat', 'long']].max().values.tolist()
m.fit_bounds([sw,ne])

# Create tool tips 
tooltips = {}
for _, row in cities.iterrows():
    city = row['city']
    tooltips[city]=f"<b>{city}</b><br><b>Population:</b> {row['pop']}"

    if city in sites_in_cities.keys():
        tooltips[city]+= "<br><b>Sites with events:</b><br>"
        tooltips[city]+= sites_in_cities[city]
    else:
        tooltips[city]+= "<br><b>No Healthwise Clinics</b>"

    if city in adjacent_sites.keys(): 
        tooltips[city]+= "<br><b>Covered by events in: </b>"
        tooltips[city]+= "<br>".join(adjacent_sites[city])


# Add cities to the map
for _, row in cities.iterrows():
    city = row['city']
    folium.CircleMarker(
        location=(row['lat'], row['long']),
        radius=get_marker_size(row['pop']),
        color="black",
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.6,
        tooltip=tooltips[city]
    ).add_to(m)

display(results)
display(m)

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%),Sites Visited,Cities covered
Solution ID,,,,,,
0,181,1601.03,3728.9,13698024.14,lima-mercy-thrift-store\nlima-lima-library\nke...,Ada\nAlger\nBluffton\nCairo\nCaledonia\nCarey\...
